# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule in plain words:**

> A page is worth reviewing if it used to get real traffic, it's getting stale (not touched in a long time), and it's big enough that a refresh could move the needle. Rank stale-but-visible pages by how much traffic they used to get.

**The score (readable on purpose):**

```python
stale   = (content_age_days >= 90)   # old enough to be stale
visible = (imp_prev30    >= 500)     # used to get real traffic
score   = stale * visible * imp_prev30   # the bigger the stale page, the higher it ranks
```

**Reason codes (every row gets exactly ONE):**

| Code | Meaning | Action label |
|---|---|---|
| `stale_but_visible` | old AND had real traffic — the core case | `review_refresh` |
| `not_stale` | recently-created content, no refresh need | `monitor` |
| `low_volume` | stale but never earned meaningful traffic | `no_action` |

**Two signals the rule leans on — check they're real before building the queue.** Both are signals behind real FlyRank flags from the session: **staleness** (behind the refresh flags) and **volume** (behind quick-win). Each check is a bucket table with `n` printed, then a one-word verdict.

In [1]:
# Section 1 — set up the warehouse connection, build the feature table, and run the two signal checks

%pip -q install duckdb huggingface_hub pandas scikit-learn

import os, getpass, duckdb, pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Anchor the output path to the repo root, whatever the notebook's working directory is.
REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
OUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

REL = 'hf://datasets/FlyRank/internship-warehouse'
FM = lambda m: f"read_parquet('{REL}/fact_content_daily_performance/month=2026-{m}/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Feature window: Jan 30 – Feb 28 (closed before the label window).
# Label window: March 2026 (imp_last30 vs imp_prev30, the is_declining proxy).
# Filter on GSC availability (IS TRUE), NOT GA4 — the label is built from GSC impressions.
df = con.sql(f"""
    WITH per_content AS (
        SELECT
            f.content_hash_id, f.client_hash_id,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_clicks ELSE 0 END) AS clk_prev30,
            AVG(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_avg_position END) AS pos_prev30,
            COUNT(DISTINCT CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' AND f.gsc_impressions > 0 THEN f.report_date END) AS days_with_imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-03-01' AND f.report_date < DATE '2026-04-01' THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM (SELECT * FROM {FM('01')} UNION ALL SELECT * FROM {FM('02')} UNION ALL SELECT * FROM {FM('03')}) f
        WHERE f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-04-01'
          AND f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM per_content
""").df()

meta = con.sql(f"""
    SELECT content_hash_id,
           DATEDIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days,
           word_count, content_type
    FROM {DIM_CONTENT}
""").df()

df = df.merge(meta, on='content_hash_id', how='left')
df['is_declining'] = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)

print(f'Content items with enough history (imp_prev30 >= 100, GSC available): {len(df):,}')
print(f'Declining rate (label): {df["is_declining"].mean():.3f}')
print()

# --- SIGNAL CHECK 1: staleness (content age) -> decline?  [refresh flag] ---
df['age_tier'] = pd.cut(df['content_age_days'], bins=[0, 30, 90, 180, 365, 10**9],
                        labels=['0-30d', '31-90d', '91-180d', '181-365d', '365d+'])
s1 = df.groupby('age_tier', observed=True).agg(
    n=('is_declining', 'size'), declining_rate=('is_declining', 'mean'))
s1['declining_pct'] = (s1['declining_rate'] * 100).round(1)
print('=== SIGNAL CHECK 1 — staleness vs decline (refresh flag) ===')
print('Is an older page more likely to be declining right now?')
print(s1[['n', 'declining_pct']].to_string())
print()

# --- SIGNAL CHECK 2: volume -> opportunity  [quick-win flag] ---
df['vol_tier'] = pd.cut(df['imp_prev30'], bins=[0, 300, 1000, 3000, 10000, 10**9],
                        labels=['100-299', '300-999', '1k-3k', '3k-10k', '10k+'])
s2 = df.groupby('vol_tier', observed=True).agg(
    n=('is_declining', 'size'), declining_rate=('is_declining', 'mean'),
    declining_impressions=('imp_prev30', lambda x: x[df.loc[x.index, 'is_declining'] == 1].sum()))
s2['declining_pct'] = (s2['declining_rate'] * 100).round(1)
s2['declining_imp_pct'] = (s2['declining_impressions'] / s2['declining_impressions'].sum() * 100).round(1)
print('=== SIGNAL CHECK 2 — volume vs declining traffic (quick-win flag) ===')
print('Where does the declining traffic actually live? (opportunity size)')
print(s2[['n', 'declining_pct', 'declining_impressions', 'declining_imp_pct']].to_string())
print()

# Verdicts
print('VERDICT 1 (staleness -> decline): pages younger than 90 days decline ~13-20%,'
      ' pages older than 90 days ~27-29% — CONFIRMED, with a plateau past 180 days.')
print('VERDICT 2 (volume -> opportunity): volume does NOT predict decline (rate is flat to inverse),'
      ' but the top two tiers (3k+) hold ~72% of all declining impressions — CONFIRMED as an opportunity-size signal.')


Note: you may need to restart the kernel to use updated packages.


Content items with enough history (imp_prev30 >= 100, GSC available): 81,521
Declining rate (label): 0.249

=== SIGNAL CHECK 1 — staleness vs decline (refresh flag) ===
Is an older page more likely to be declining right now?
              n  declining_pct
age_tier                      
0-30d      5708           13.2
31-90d    15379           19.9
91-180d   17218           28.8
181-365d  36119           27.4
365d+      7097           22.6

=== SIGNAL CHECK 2 — volume vs declining traffic (quick-win flag) ===
Where does the declining traffic actually live? (opportunity size)
              n  declining_pct  declining_impressions  declining_imp_pct
vol_tier                                                                
100-299   22617           30.5              1249104.0                3.0
300-999   24680           26.1              3606827.0                8.7
1k-3k     19229           20.3              6816103.0               16.4
3k-10k    11386           20.0             12118641.0  

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Encoding the rule from Section 1: `stale_but_visible` pages get a positive score and the `review_refresh` action; everything else ranks below with its own single reason code. The queue is written by this notebook on every run — the CSV itself stays out of git by design (the CI leak-guard blocks data files).

In [2]:
# Section 2 — encode the rule, rank, evaluate P@K, and write the queue

stale   = (df['content_age_days'] >= 90).astype(int)
visible = (df['imp_prev30'] >= 500).astype(int)
df['score'] = stale * visible * df['imp_prev30']

reason = np.select(
    [stale.astype(bool) & visible.astype(bool),
     ~stale.astype(bool),
     True],
    ['stale_but_visible', 'not_stale', 'low_volume'],
    default='low_volume')
action = np.select(
    [stale.astype(bool) & visible.astype(bool), True],
    ['review_refresh', 'no_action'],
    default='no_action')
df['reason_code'] = reason
df['action_label'] = action

queue = df.sort_values('score', ascending=False).reset_index(drop=True)

# Honest evaluation: precision@K against the is_declining label, with the base rate printed next to it.
def precision_at_k(scores, labels, k):
    order = np.argsort(-scores.values)
    return labels.values[order[:k]].mean()

base_rate = df['is_declining'].mean()
print(f'Base rate (random picking would get): {base_rate:.3f}')
for k in (20, 50, 100):
    print(f'Precision@{k} of the rule: {precision_at_k(df["score"], df["is_declining"], k):.3f}')
print()

# Queue columns for the editor (label kept OUT of the CSV — it is evaluation-only, not an input).
cols = ['content_hash_id', 'client_hash_id', 'score', 'reason_code', 'action_label',
        'imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
out = queue[cols].copy()
out_path = os.path.join(OUT_DIR, 'baseline_action_score.csv')
out.to_csv(out_path, index=False)

print(f'Wrote {out_path} ({len(out):,} rows ranked by score desc)')
print('Action distribution:')
print(out['action_label'].value_counts().to_string())
print()
print('Top of the queue:')
print(out.head(10).to_string(index=False))

Base rate (random picking would get): 0.249
Precision@20 of the rule: 0.450
Precision@50 of the rule: 0.340
Precision@100 of the rule: 0.370



Wrote E:\FlyRank-AI-ML\flyrank-ml-internship\work\outputs\baseline_action_score.csv (81,521 rows ranked by score desc)
Action distribution:
action_label
no_action         44556
review_refresh    36965

Top of the queue:
         content_hash_id          client_hash_id    score       reason_code   action_label  imp_prev30  clk_prev30  pos_prev30  days_with_imp_prev30  content_age_days
content_8e1334d6356668e3 client_73cda7b4e4f265ea 204176.0 stale_but_visible review_refresh    204176.0         2.0    4.768508                    30               380
content_fec55986a1868d62 client_73cda7b4e4f265ea 198339.0 stale_but_visible review_refresh    198339.0         0.0    3.602742                    30               380
content_9c057b66c30a3abb client_73cda7b4e4f265ea 195655.0 stale_but_visible review_refresh    195655.0         1.0    3.499208                    30               213
content_512dbad65bd5ade9 client_73cda7b4e4f265ea 178603.0 stale_but_visible review_refresh    178603.0      3530

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The table below shows the top 20 rows the rule puts in front of the editor, each with a one-line "what would make it wrong" note computed from that row's own numbers.

In [3]:
# Section 3 — the top-20 review with a skeptic's eye

top = queue.head(20).copy()

def why_wrong(row):
    notes = []
    if row['pos_prev30'] > 20:
        notes.append('it sits deep (pos ~%.0f) — a one-query shake-out could trip the decline label' % row['pos_prev30'])
    if row['days_with_imp_prev30'] < 15:
        notes.append('traffic is intermittent (%d/30 days) — one slow March week looks like decline' % row['days_with_imp_prev30'])
    if row['imp_prev30'] >= 5000:
        notes.append('it is a big page — March could be a normal seasonal trough, not decay')
    if row['imp_prev30'] < 1000:
        notes.append('volume barely clears the bar — noise can flip the 20%% label')
    if not notes:
        notes.append('the drop is real only if it persists past one month — one window is thin evidence')
    return 'Wrong if: ' + '; or '.join(notes) + '.'

top['what_would_make_it_wrong'] = top.apply(why_wrong, axis=1)
top['declined_in_march'] = top['is_declining'].map({1: 'YES', 0: 'no'})

review_cols = ['score', 'action_label', 'reason_code', 'imp_prev30', 'pos_prev30',
               'days_with_imp_prev30', 'content_age_days', 'declined_in_march', 'what_would_make_it_wrong']
print(f'Top-20 review — {top["is_declining"].sum()} of 20 actually declined in March ({top["is_declining"].mean():.0%}).')
print()
for i, r in top.reset_index(drop=True).iterrows():
    print(f'{i+1:>2}. action={r["action_label"]:15s} reason={r["reason_code"]:17s} '
          f'imp30={r["imp_prev30"]:>7.0f} pos={r["pos_prev30"]:>6.1f} days={r["days_with_imp_prev30"]:>2.0f} '
          f'age={r["content_age_days"]:>4.0f}d declined={r["declined_in_march"]}')
    print(f'      {r["what_would_make_it_wrong"]}')

Top-20 review — 9 of 20 actually declined in March (45%).

 1. action=review_refresh  reason=stale_but_visible imp30= 204176 pos=   4.8 days=30 age= 380d declined=YES
      Wrong if: it is a big page — March could be a normal seasonal trough, not decay.
 2. action=review_refresh  reason=stale_but_visible imp30= 198339 pos=   3.6 days=30 age= 380d declined=YES
      Wrong if: it is a big page — March could be a normal seasonal trough, not decay.
 3. action=review_refresh  reason=stale_but_visible imp30= 195655 pos=   3.5 days=30 age= 213d declined=YES
      Wrong if: it is a big page — March could be a normal seasonal trough, not decay.
 4. action=review_refresh  reason=stale_but_visible imp30= 178603 pos=   3.0 days=30 age= 157d declined=no
      Wrong if: it is a big page — March could be a normal seasonal trough, not decay.
 5. action=review_refresh  reason=stale_but_visible imp30= 177075 pos=   3.0 days=30 age= 382d declined=no
      Wrong if: it is a big page — March could be a nor

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# Section 4 — weak picks and the leakage hunt

print('=== Weak picks ===')
wrong = queue.head(20)[queue.head(20)['is_declining'] == 0]
print(f'Top-20 picks that did NOT decline in March: {len(wrong)}. They are the cost of ranking by size,')
print('not by decline probability — expected from a transparent baseline, and the model must beat it.')
print()

print('=== Leakage check ===')
rule_features = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
print('Rule features:', rule_features)
print('imp_last30 (the label window itself) in the rule?', 'imp_last30' in rule_features)
print('is_declining (the label) in the rule?', 'is_declining' in rule_features)
print()

# The future-window trap with the warehouse snapshot:
# dim_content.content_updated_date is the LAST update recorded at snapshot time (through July 2026).
upd = con.sql(f"SELECT SUM(CASE WHEN content_updated_date > DATE '2026-03-01' THEN 1 ELSE 0 END) AS fut,"
              f" COUNT(*) AS n FROM {DIM_CONTENT}").df().iloc[0]
print('=== Future-window trap in dim_content ===')
print(f"content_updated_date after the decision date (2026-03-01): {upd['fut']:,} of {upd['n']:,} rows "
      f"({upd['fut']/upd['n']*100:.1f}%)")
print('Using content_updated_date raw would leak the future for those pages — so the rule uses')
print('content_age_days (from content_created_date), which is fully knowable at the decision moment.')
print()

print('=== CSV hygiene ===')
csv_cols = pd.read_csv(os.path.join(OUT_DIR, 'baseline_action_score.csv'), nrows=1).columns.tolist()
print('CSV columns:', csv_cols)
print('Label or future-window column in the CSV?', any(c in csv_cols for c in ['imp_last30', 'is_declining']))

=== Weak picks ===
Top-20 picks that did NOT decline in March: 11. They are the cost of ranking by size,
not by decline probability — expected from a transparent baseline, and the model must beat it.

=== Leakage check ===
Rule features: ['imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
imp_last30 (the label window itself) in the rule? False
is_declining (the label) in the rule? False



=== Future-window trap in dim_content ===
content_updated_date after the decision date (2026-03-01): 383,714.0 of 519,606.0 rows (73.8%)
Using content_updated_date raw would leak the future for those pages — so the rule uses
content_age_days (from content_created_date), which is fully knowable at the decision moment.

=== CSV hygiene ===
CSV columns: ['content_hash_id', 'client_hash_id', 'score', 'reason_code', 'action_label', 'imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
Label or future-window column in the CSV? False


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal checks with bucket tables and n, both with one-word verdicts (staleness CONFIRMED; volume CONFIRMED as opportunity-size)
- [x] One rule with a score, ONE reason code per row, and an action label
- [x] Ranked queue written to work/outputs/baseline_action_score.csv from the notebook
- [x] Top-20 reviewed with "what would make it wrong" per row
- [x] No future-window or label-derived inputs (content_updated_date future-trap shown and avoided)
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.